# 04-02 - Data Cleaning & Fixing

**Phase:** 04 - Data Analysis & Preparation

**Difficulty:** 1/3 | **Priority:** 1/2

**Status:** VERIFIED

---

## 1. What Are We Solving?

Real-world data is messy: inconsistent formats, duplicates, typos, and errors. **Data cleaning** is the process of fixing these issues so the data is reliable for analysis and modeling.

## 2. Why Does This Matter?

Garbage in, garbage out. A model trained on dirty data produces unreliable results. Cleaning is often 60-80% of a data scientist's work.

## 3. Prerequisites

- Unit 04.1 (EDA)
- Phase 01 (Python Foundations)

## 4. Learning Objectives

By the end of this notebook, you should be able to:
- [ ] Detect and remove duplicates
- [ ] Standardize inconsistent formats
- [ ] Fix typos and inconsistent categories
- [ ] Handle whitespace and case issues
- [ ] Convert data types correctly
## 5. Mental Model

**Mental Model:** Data cleaning is like washing vegetables before cooking. You wouldn't throw dirty vegetables into a pot — you'd rinse off the dirt, remove the wilted leaves, and trim the bad parts. Data cleaning does the same: remove junk, fix errors, and prepare clean ingredients for your model.

Key: understand the data before you model it.


## 2a. Decision Guidance

| Situation | What to Do | Why |
|-----------|------------|-----|
| Mixed data types in one column | Parse or coerce types | Prevents downstream errors |
| Trailing whitespace in strings | Use `.str.strip()` | Prevents mismatched categories |
| Inconsistent date formats | Use `pd.to_datetime()` with format | Enables time-based analysis |
| Duplicate rows | Check with `.duplicated()` | Prevents inflated counts |
| Garbage characters in numeric columns | Coerce to NaN, then investigate | Clean numeric data |


## 2b. Common Mistakes to Avoid

- Cleaning only the training set (not applying same steps to test)
- Over-cleaning: removing too much data
- Not documenting cleaning decisions
- Changing data without understanding why
- Forgetting to re-run cleaning after new data arrives


## 6. Create a Messy Dataset

Let's create a dataset with common problems: duplicates, inconsistent case, typos, whitespace, and wrong types.


In [1]:
import pandas as pd
import numpy as np

# Messy dataset
data = {
    "name": ["Alice", "bob", "Bob", "  Charlie  ", "alice", "David", "Eve", "bob"],
    "city": ["New York", "new york", "NYC", "Los Angeles", "New York", "LA", "los angeles", "New York"],
    "age": ["25", "30", "30", "28", "25", "35", "29", "30"],
    "salary": [50000, 60000, 60000, 55000, 50000, 70000, 58000, 60000],
}
df = pd.DataFrame(data)
print("Messy dataset:")
print(df)


Messy dataset:
          name         city age  salary
0        Alice     New York  25   50000
1          bob     new york  30   60000
2          Bob          NYC  30   60000
3    Charlie    Los Angeles  28   55000
4        alice     New York  25   50000
5        David           LA  35   70000
6          Eve  los angeles  29   58000
7          bob     New York  30   60000


## 7. Detect Duplicates

Duplicates inflate the data and bias results. Let's find and remove them.


In [2]:
# Detect duplicates
print("Duplicate rows:")
print(df[df.duplicated()])
print(f"\nNumber of duplicate rows: {df.duplicated().sum()}")

# Remove exact duplicates
df_clean = df.drop_duplicates().copy()
print(f"\nAfter removing duplicates: {len(df_clean)} rows (from {len(df)})")


Duplicate rows:
Empty DataFrame
Columns: [name, city, age, salary]
Index: []

Number of duplicate rows: 0

After removing duplicates: 8 rows (from 8)


## 8. Standardize Case and Whitespace

Inconsistent case ('bob' vs 'Bob') and whitespace ('  Charlie  ') make categories look different when they're the same.


In [3]:
# Standardize name: strip whitespace and title case
df_clean["name"] = df_clean["name"].str.strip().str.title()
print("Names after cleaning:")
print(df_clean["name"].tolist())

# Standardize city: lowercase for comparison
df_clean["city_lower"] = df_clean["city"].str.strip().str.lower()
print("\nCities (lowercased):")
print(df_clean["city_lower"].tolist())


Names after cleaning:
['Alice', 'Bob', 'Bob', 'Charlie', 'Alice', 'David', 'Eve', 'Bob']

Cities (lowercased):
['new york', 'new york', 'nyc', 'los angeles', 'new york', 'la', 'los angeles', 'new york']


## 9. Fix Inconsistent Categories

The same city appears as 'New York', 'new york', and 'NYC'. We need to map them to one canonical value.


In [4]:
# Map city aliases to canonical values
city_map = {
    "new york": "New York",
    "nyc": "New York",
    "los angeles": "Los Angeles",
    "la": "Los Angeles",
}
df_clean["city"] = df_clean["city_lower"].map(city_map)
print("Cities after mapping:")
print(df_clean["city"].tolist())
print("\nNow each city has one consistent value.")


Cities after mapping:
['New York', 'New York', 'New York', 'Los Angeles', 'New York', 'Los Angeles', 'Los Angeles', 'New York']

Now each city has one consistent value.


## 10. Convert Data Types

The 'age' column is stored as strings. We need to convert it to integers.


In [5]:
# Convert age from string to int
print("Age dtype before:", df_clean["age"].dtype)
df_clean["age"] = df_clean["age"].astype(int)
print("Age dtype after:", df_clean["age"].dtype)
print("\nCleaned dataset:")
print(df_clean[["name", "city", "age", "salary"]])


Age dtype before: str
Age dtype after: int64

Cleaned dataset:
      name         city  age  salary
0    Alice     New York   25   50000
1      Bob     New York   30   60000
2      Bob     New York   30   60000
3  Charlie  Los Angeles   28   55000
4    Alice     New York   25   50000
5    David  Los Angeles   35   70000
6      Eve  Los Angeles   29   58000
7      Bob     New York   30   60000


## 11. Handle Typos

Typos create spurious categories. Let's look at a column with typos and fix them.


In [6]:
# Dataset with typos in a category column
df2 = pd.DataFrame({
    "product": ["laptop", "Laptop", "laptp", "phone", "Phone", "phne", "tablet", "Tablet"],
    "price": [1000, 1000, 1000, 500, 500, 500, 300, 300],
})
print("Products with typos:")
print(df2["product"].value_counts())

# Fix typos and case
typo_map = {"laptp": "laptop", "phne": "phone"}
df2["product"] = df2["product"].str.lower().replace(typo_map)
print("\nProducts after fixing typos:")
print(df2["product"].value_counts())
print("\nNow there are 3 clean categories instead of 8 messy ones.")


Products with typos:
product
laptop    1
Laptop    1
laptp     1
phone     1
Phone     1
phne      1
tablet    1
Tablet    1
Name: count, dtype: int64

Products after fixing typos:
product
laptop    3
phone     3
tablet    2
Name: count, dtype: int64

Now there are 3 clean categories instead of 8 messy ones.


## 12. Failure Case: Not Cleaning

If you don't clean, 'bob' and 'Bob' count as different people, and 'NYC' and 'New York' as different cities. This inflates category counts and biases analysis.


In [7]:
# Show the impact of not cleaning
print("UN-CLEANED city counts (looks like 5 cities):")
print(df["city"].value_counts())
print("\nCLEANED city counts (actually 2 cities):")
print(df_clean["city"].value_counts())
print("\nCleaning reveals the true structure of the data.")


UN-CLEANED city counts (looks like 5 cities):
city
New York       3
new york       1
NYC            1
Los Angeles    1
LA             1
los angeles    1
Name: count, dtype: int64

CLEANED city counts (actually 2 cities):
city
New York       5
Los Angeles    3
Name: count, dtype: int64

Cleaning reveals the true structure of the data.


## 13. Debugging: Common Errors

- **Not stripping whitespace**: 'Bob' vs ' Bob'.
- **Case sensitivity**: 'bob' vs 'Bob'.
- **Aliases**: 'NYC' vs 'New York'.
- **Wrong types**: numbers stored as strings.
- **Not checking after cleaning**: verify the result.

## 14. Real-World Considerations

- Cleaning is iterative - you'll discover issues as you go.
- Keep a record of cleaning steps for reproducibility.
- Don't over-clean - removing too much data loses information.
- Validate cleaning with EDA after each step.

## 15. Common Mistakes

- Dropping rows with fixable issues.
- Not standardizing before deduplication.
- Ignoring whitespace and case.
- Not verifying the cleaned result.

## 16. When NOT to Use

- Don't remove data you can fix.
- Don't apply aggressive cleaning without understanding the domain.
- Don't clean blindly - verify each step.

## 17. Challenge

Clean a dataset with duplicate names, inconsistent case, and a typo in the category column.


In [8]:
# Challenge: clean a messy dataset
messy = pd.DataFrame({
    "user": ["John", "john", "JOHN", "  Mary  ", "mary", "John"],
    "status": ["active", "Active", "active", "inactive", "inactiv", "active"],
    "score": ["85", "85", "85", "92", "92", "85"],
})

# Step 1: strip + lowercase user, then title
messy["user"] = messy["user"].str.strip().str.title()

# Step 2: fix typo in status
messy["status"] = messy["status"].str.lower().replace({"inactiv": "inactive"})

# Step 3: convert score to int
messy["score"] = messy["score"].astype(int)

# Step 4: drop duplicates
messy = messy.drop_duplicates()

print("Cleaned challenge dataset:")
print(messy)
print(f"\nUnique users: {messy['user'].nunique()}")
print(f"Unique statuses: {messy['status'].unique().tolist()}")
print(f"Score dtype: {messy['score'].dtype}")


Cleaned challenge dataset:
   user    status  score
0  John    active     85
3  Mary  inactive     92

Unique users: 2
Unique statuses: ['active', 'inactive']
Score dtype: int64


## 19. Knowledge Check

1. What is the difference between `.dropna()` and `.fillna()`? When would you use each?
2. Why should you apply the same cleaning steps to both training and test sets?
3. What is the danger of cleaning data without understanding the context?
4. How would you handle a column that has mixed types (e.g., numbers and 'N/A' strings)?
5. Why is it important to document your cleaning decisions?

## 19. Teach-Back Questions

Explain to another person:

- Why cleaning is the most time-consuming part of data work.
- How to standardize a messy category column.
- Why you should verify after each cleaning step.

## 20. Summary

You now know how to clean data: remove duplicates, standardize case and whitespace, fix inconsistent categories and typos, and convert data types. Clean data is the foundation of reliable analysis and modeling.


## 21a. Exit Criteria

- [ ] I can identify data type issues with `df.info()`
- [ ] I can fix string columns (strip, lowercase, replace)
- [ ] I can convert columns to appropriate types
- [ ] I can find and remove duplicate rows
- [ ] I can apply cleaning steps consistently to new data

## 21b. Next Step

Proceed to `04_03_missing_values.ipynb` to learn how to handle missing data.

## 22. Hands-On Practice

**Level 1 - Observation:** Run `df.info()` to identify data type issues.

**Level 2 - Guided:** Fix one string column with `.str.strip()` and `.str.lower()`.

**Level 3 - Practice:** Convert a column to datetime using `pd.to_datetime()` with error handling.

**Level 4 - Challenge:** Find and remove all duplicate rows, documenting how many were removed.

**Level 5 - Mastery:** Write a cleaning pipeline that handles type conversion, duplicates, and whitespace in sequence.

## 22. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: pandas, numpy
OUTPUTS: PASS
LAST VERIFIED: 2026-08-28
```
